In [193]:
# otter configuration
# (no code needed here if using the CLI; this is just a marker)

# PA2: Image Processing Fundamentals

## Overview
This assignment introduces fundamental concepts in signal processing and array manipulation through practical image processing implementations.

## Learning Objectives
After completing this assignment, you will be able to:
1. Represent and manipulate digital images using NumPy arrays
2. Implement common image processing algorithms from first principles
3. Apply convolution operations to perform image filtering
4. Understand how basic image enhancement techniques work

## Assignment Tasks
You will implement various image processing algorithms including:
- White balance correction
- Contrast adjustment and histogram equalization
- Image filtering via convolution (smoothing, 3D effects, edge detection)

## Prerequisites
- Basic knowledge of NumPy array operations
- Understanding of for loops and array indexing
- Familiarity with RGB color representation

## Due Date
Due in 2 weeks: Monday, 10/16/2025 by 5pm.

Use Slack to discuss problems and ask questions.


## Working with images

Code below will import an image to a numpy array and discuss how it is stored in memory.

**Note for Students:**
This assignment requires image files that you can obtain in two ways:
1. Use the local files provided in your course materials.
2. Use Google Drive to access the images if you don't have the files locally.

To use the Google Drive option:
- Uncomment the appropriate lines in the code cells
- Replace the placeholder file IDs with the actual Google Drive file IDs provided by your instructor
- The images will be downloaded automatically when you run the notebook

In [194]:
# Run this cell first to install required packages
try:
    # Check if we're running in Colab
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing required packages...")
    !pip install -q numpy matplotlib requests
except ImportError:
    IN_COLAB = False
    print("Running in local Jupyter environment.")

%matplotlib notebook

import numpy as np
import matplotlib.pyplot as plt
print("Setup complete!")

Running in local Jupyter environment.
Setup complete!


In [195]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Function to load image data from Google Drive or local directory
def load_image(file_id, filename):
    """
    Load image data from Google Drive (works in both Colab and regular notebooks)

    Parameters:
    -----------
    file_id : str
        Google Drive file ID (from the shared link)
    filename : str
        Filename to save locally

    Returns:
    --------
    img : numpy array
        The image as a numpy array
    """
    try:
        # First check if file already exists locally
        if os.path.exists(filename):
            print(f"Loading {filename} from local directory...")
            return mpimg.imread(filename)

        # Check if running in Colab
        in_colab = 'google.colab' in str(get_ipython())

        if in_colab:
            # Use gdown in Colab (simpler approach)
            print(f"Downloading {filename} from Google Drive...")
            !pip install -q gdown
            !gdown {file_id} -O {filename}
        else:
            # For regular Jupyter notebooks, download directly using requests
            print(f"Downloading {filename} from Google Drive...")
            import requests

            # Create the download URL from the file ID
            url = f"https://drive.google.com/uc?id={file_id}"

            # Download the file
            response = requests.get(url)
            with open(filename, 'wb') as f:
                f.write(response.content)

        # Load and return the image
        print(f"Successfully downloaded {filename}")
        return mpimg.imread(filename)

    except Exception as e:
        print(f"Error loading image: {e}")

        # If all else fails, raise the error
        raise FileNotFoundError(f"Could not load {filename}. Please check your internet connection or file path.")

In [196]:
# Image file IDs in Google Drive - Replace these with the actual file IDs for your images
# You can get the file ID from the Google Drive sharing link
# Example: https://drive.google.com/file/d/FILE_ID_GOES_HERE/view?usp=sharing
image_file_ids = {
    'pic0.jpg': '1AxZYKkq4qRsjOyEARMj0RxVKPLukBePV',
    'wit_help.jpg': '1B3UInU9jna7_MjhVunEck7FpDXzIq51O'
}


# https://drive.google.com/file/d/1AxZYKkq4qRsjOyEARMj0RxVKPLukBePV/view?usp=sharing
# https://drive.google.com/file/d/1B3UInU9jna7_MjhVunEck7FpDXzIq51O/view?usp=sharing

# Load the necessary images (uncomment these lines and update the file IDs to use them)
img_boston = load_image(image_file_ids['pic0.jpg'], 'pic0.jpg')
img_wit = load_image(image_file_ids['wit_help.jpg'], 'wit_help.jpg')

Loading pic0.jpg from local directory...
Loading wit_help.jpg from local directory...


In [197]:
#import goodies
%matplotlib notebook

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import numpy as np

In [198]:
#Read an image of Boston:
fname = 'pic0.jpg'
# Use the load_image function if loading from Google Drive
img = load_image(image_file_ids['pic0.jpg'], fname)
# Or use the traditional method if the file is available locally
# img = mpimg.imread(fname)
#image courtesy of pixabay: https://pixabay.com/en/boston-water-front-city-1448339/

Loading pic0.jpg from local directory...


In [199]:
#display the image as a 2D array
import cv2
image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.imshow("temp", image)

# plt.figure()
# plt.imshow(img)
# plt.show()


In [200]:
print(img.shape)
# Image is MxNx3


(856, 1280, 3)


In [201]:
print(img.dtype)

uint8


Image is stored into an ndarray using  unsigned 8-bit ints. Its shape is 856x1280x3.

There are 856x1280 pixels.

Each pixel has a value ranging from 0 to 255.

The array depth (3) correspond to the RGB color space.

(R)ed, (G)reen and (B)lue are primary colors and are combined to create other colors.

Thus, we need 24 bits of data (8 bits of R + 8 bits of G + 8 bits of B) per pixel.

## White balance

We have the photo stored as a 3D array. Now we can process it!

Notice that the color scheme is a little off.

Let's develop an algorithm to auto-correct it.

We generally normalize the image with respect to (G)reen for white balance.

Algo outline:
1. Sum each color in photo
2. Scale the other colors sum with respect to (G)reen sum
3. Adjust the pixels using the new gains
4. Normalize the array and store as original type (unit8)

The code below executes steps 1 - 3. You need to complete step 4.

In [202]:
# 1. Compute the sum for each color
R = np.sum(img[:,:,0])
G = np.sum(img[:,:,1])
B = np.sum(img[:,:,2])


In [203]:
# 2. Calculate the gain factor with respect to (wrt) G:
kr = G/R
kb = G/B
print("kr = ", str(kr),", kb = ", str(kb))

kr =  0.7495015652220238 , kb =  0.45926704975480354


If the image is too blue, sum of the blue pixels will be larger than the sum of the red pixels.

That means that kb (blue correction gain factor) will be smaller.

Multiply the image by these gains and display the result.

In [204]:
# 3. Update the RGB values using the new gains
Rw = img[:,:,0] * kr
Gw = img[:,:,1] * 1.
Bw = img[:,:,2] * kb

Final step: We need to combine these 3 matrices (Rw, Gw and Bw) into a single 3D matrix and convert it unit8 to save.

No `for` loops are needed.

In [205]:
def f2int(array0, array1, array2):
    ''' Normalize the RGB array and convert to uint8.
    Find the array's max element and normalize the array values to range from 0 to 1.
    Then, scale the array for values to range from 0 to 255.
    Finally return the array as unit8.
    In: array0: 1D matrix corresponding to corrected R values
    In: array1: 1D matrix corresponding to corrected G values
    In: array2: 1D matrix corresponding to corrected B values
    Out: array: 3D matrix as unit8
    '''
    # combine arrays into a matrix
    array = np.dstack((array0, array1, array2))
    
    # find max val in the entire array
    max_val = np.max(array)
    
    # avoid div by zero errors
    if max_val == 0:
        return array.astype(np.uint8)
        
    # normalize data into a range of 0 to 1
    array_norm = array / max_val
    
    # scale data on a range from 0 to 255
    array_scaled = array_norm * 255
    
    # convert arr to unsigned 8 bit int and return
    return array_scaled.astype(np.uint8)

After the gain adjustment, the picture should be more balanced!

If you take a picture under incandescent bulbs that cause a yellowish tint, this type of algorithm should help to balance the photo.

if the input is

```
f2int([0 1 2], [0 1 2], [0 1 2])
```
the resulting array should be:
```
[[[  0   0   0]
  [127 127 127]
  [255 255 255]]]
```

In [206]:
#test code
# load input and check white balance
imwb = f2int(Rw, Gw, Bw)

In [207]:
plt.figure()
imgplot = plt.imshow(imwb)

#you can save the image as:
#plt.imsave('imwb.jpg', imwb)

<IPython.core.display.Javascript object>

In [208]:
# === PUBLIC TESTS: f2int ===
import numpy as np

# Small synthetic channels with a known global max
r = np.array([[0.0, 2.0],
              [1.0, 0.0]])
g = np.array([[0.0, 1.0],
              [0.5, 0.2]])
b = np.array([[0.5, 0.1],
              [0.2, 0.0]])

stu = f2int(r, g, b)

# Basic shape / dtype checks
assert isinstance(stu, np.ndarray), "Output must be a numpy array"
assert stu.dtype == np.uint8, "Output dtype must be uint8"
assert stu.shape == (2, 2, 3), "Output must have shape (H, W, 3)"

# Correct scaling against a reference implementation
stack = np.dstack([r, g, b])
ref = (stack / stack.max() * 255).astype(np.uint8)

assert np.array_equal(stu, ref), "Values do not match expected normalization/scaling"

print("✅ f2int public tests passed!")


✅ f2int public tests passed!


## Converting an image to grayscale

Before we do further imge processing, we will discuss how to convert a color image to grayscale.

Many algorithms reduce computational time by reducing the image depth from 3 to 1. Some even increase their accuracy running on grayscale images.

There are many methods to convert a color image to grayscale that include different coefficients.  

One common forumula given by __[Wiki](https://en.wikipedia.org/wiki/Grayscale#Converting_color_to_grayscale)__ is:

$$ Y_\mathrm{linear} = 0.2126 R_\mathrm{linear} + 0.7152 G_\mathrm{linear} + 0.0722 B_\mathrm{linear} $$

These coefficients are specified in ITU-R BT.709 and employed for high-definition television.

We can convert images to black and white using the constants given above.

Let's look at an image from WIT's social media from a site visit in Spring 2018 at the Lamar High School project in Houston, TX.

In [209]:
#let us read an image:
fname = 'wit_help.jpg'
# Use the load_image function if loading from Google Drive
# img = load_image(image_file_ids['wit_help.jpg'], fname)
# Or use the traditional method if the file is available locally
img = mpimg.imread(fname)

#image courtesy of WIT insta: https://www.instagram.com/p/BgDQw1igPAZ/?taken-by=wentworthinstitute
plt.figure()
imgplot = plt.imshow(img[:,:,:])

<IPython.core.display.Javascript object>

Below is the conversion algorithm using an inefficient for loop:

In [210]:
def rgb2bw_for(photo):
    #multiply each pixel intensity by the given coeff.
    #return as unit8
    #this is probably not the fastest way to do it.
    kr = 0.2126; kg = 0.7152; kb = 0.0722
    m,n,d = photo.shape
    bw = np.zeros((m,n))
    for i in range(m):
        for j in range(n):
            bw[i,j] = photo[i,j,0] * kr + photo[i,j,1] * kg + photo[i,j,2] * kb
    return bw.astype(np.uint8)

Let's convert our image to grayscale and display it.

Use slices and vectorized operations for all the pixels at once

In [211]:
def rgb2bw(photo):
    """Convert RGB image to grayscale
    In: photo: 3D array representing an RGB color image
    Out: 2D array representing a grayscale image
    """
    kr = 0.2126; kg = 0.7152; kb = 0.0722
    # Formula: Y = 0.2126*R + 0.7152*G + 0.0722*B
    
    # use np dot as a vectorized & weighted sum across color channels
    grayscale_image = np.dot(photo[...,:3], [kr, kg, kb])
    # convert array to uintt8 and return
    return grayscale_image.astype(np.uint8)

In [212]:
# === PUBLIC TESTS: rgb2bw ===
import numpy as np

# Random small RGB image (uint8)
np.random.seed(0)
photo = np.random.randint(0, 256, size=(4, 5, 3), dtype=np.uint8)

# Expected grayscale using the standard luminance coefficients
kr, kg, kb = 0.2126, 0.7152, 0.0722
ref = (photo[:, :, 0] * kr + photo[:, :, 1] * kg + photo[:, :, 2] * kb).astype(np.uint8)

stu = rgb2bw(photo)

assert isinstance(stu, np.ndarray), "Output must be a numpy array"
assert stu.dtype == np.uint8, "Output dtype must be uint8"
assert stu.shape == (4, 5), "Output must be a 2D grayscale image"

# Value check (exact equality is fine because both use uint8 cast)
assert np.array_equal(stu, ref), "Grayscale values do not match expected luminance formula"

print("✅ rgb2bw public tests passed!")

✅ rgb2bw public tests passed!


Let's check if our implementation works!

The implementation should be **much faster** than the non-vectorized version.

Let's first visually check that the results look similar (these are approximate, and so they may not be exactly identical due to small differences in floating-point arithmetic):

1. Time the executions
2. Display the images
3. Compare the max difference between the 2 implementations

In [213]:
import time

t0 = time.time()
ph1 = rgb2bw_for(img)
print("time for loop: ", time.time()-t0)

time for loop:  2.3102283477783203


In [214]:
t0 = time.time()
ph2 = rgb2bw(img)
print("time vectorized: ", time.time()-t0)

print("Max diff: ", np.max(np.abs(ph2.astype(float) - ph1.astype(float))))

time vectorized:  0.008768558502197266
Max diff:  1.0


In [215]:
def display_images(im1, im2, title1='Original', title2='Processed'):
    '''Displays two images side by side for visual comparison.
    This is much prettier than using subplot
    '''
    import matplotlib.gridspec as gridspec

    plt.figure(figsize=(14,7))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])

    #subplot 1
    ax0 = plt.subplot(gs[0])
    ax0.imshow(im1, cmap='gray')
    ax0.set_title(title1)

    #subplot 2
    ax1 = plt.subplot(gs[1])
    ax1.imshow(im2, cmap='gray')
    ax1.set_title(title2)

    plt.tight_layout()
    plt.show()

In [216]:
# Check that output is correct
display_images(ph1, ph2, "For Loop Implementation", "Vectorized Implementation")

print(ph1.shape, ph2.shape)
print(ph1.dtype, ph2.dtype)

<IPython.core.display.Javascript object>

(732, 1080) (732, 1080)
uint8 uint8


In [217]:
# ph1, ph2 look very similar, except the computation time is much faster for the vectorized version

In [218]:
# Let us compute the histogram of the image (to be used in equalization)
import numpy as np

In [219]:
hist = np.zeros(256)

for i in range(256):
    hist[i] = np.sum(ph2==i)

plt.figure(figsize=(10,5))
plt.bar(np.arange(256), hist)
plt.title('Grayscale histogram')
plt.xlabel('Pixel value')
plt.ylabel('Frequency')
plt.xlim([0, 256])

<IPython.core.display.Javascript object>

(0.0, 256.0)

Notice that the most pixels have low values.

Let's look at another synthetic test image that is mostly black, with a small visible portion.

## Histogram Equalization

If a grayscale image has low contrast, we can enhance it with histogram equalization.

This is a common technique that remaps the pixel intensities to enhance contrast.

The idea: transform the pixel values in the original such that the values in the resulting histogram will be uniformly distributed.

This equalization applies to grayscale images. For color images, we have to be careful on how to apply the algorithm to prevent color distortion.  

You can find more information about this algorithm on [wiki](https://en.wikipedia.org/wiki/Histogram_equalization).

In [220]:
from matplotlib.colors import Normalize
norm = Normalize(vmin=0, vmax=255)
img_dark = np.zeros((100,100), dtype='uint8')
img_dark[20:40, 30:70] = 100

plt.imshow(img_dark, cmap='gray', norm=norm)  # Set colormap to 'gray' for grayscale images
plt.colorbar()
plt.title("Synthetic test image")

<IPython.core.display.Javascript object>

Text(0.5, 1.0, 'Synthetic test image')

Let's implement histogram equalization to improve the image contrast.

Basically, we want to transform this image so that its histogram will be uniformly distributed.

Histogram equalization algorithm:

1. Calculate histogram of the original image as $h(g)$.
2. Calculate normalized cumulative distribution function (CDF) as $cdf(g) = \sum_{i=0}^{g} \frac{h(i)}{N} $ where $N$ is total number of pixels
3. Transform pixel values $g$ using the normalized CDF as $g_{out} = cdf(g) \cdot (L-1)$ where $L$ is number of intensity levels (e.g., 256 for 8-bit images)

In [221]:
# Let's check the histogram first
hist_dark = np.zeros(256)
for i in range(256):
    hist_dark[i] = np.sum(img_dark == i)

Now, let's implement the equalization algorithm:

In [222]:
def histeq(im):
    """Histogram equalization to enhance contrast
    In: im: grayscale image
    Out: eqim: histogram-equalized image
    """
    # im is already a 2D grayscale uint8 image

    # step 1, flatten turns image into an array, bins are set to 256 for the intensity level range of 0 to 255
    hist, bins = np.histogram(im.flatten(), bins=256, range=[0,256])
    
    # step 2, calculate CDF, which is the cumulative sum of the histogram.
    cdf = hist.cumsum()

    # step 3, create mapping function via normalizing CDF, masking 0s to avoid div by 0 and to ensure min val is mapped to 0
    cdf_masked = np.ma.masked_equal(cdf, 0)
    
    # normalize histogram, scaling CDF to 0-255. this creates the LUT.
    lut = (cdf_masked - cdf_masked.min()) * 255 / (cdf_masked.max() - cdf_masked.min())
    
    # fill masked values
    lut = np.ma.filled(lut, 0).astype('uint8')
    
    
    # step 4, apply mapping to the original image to equalize the image
    eqim = lut[im]
    
    return eqim

In [223]:
# === PUBLIC TESTS: histeq ===
import numpy as np

# Construct a simple bimodal image: many bright pixels, fewer dark pixels
im = np.full((8, 8), 200, dtype=np.uint8)
im[:4, :4] = 10  # dark quadrant

eq = histeq(im)

# Basic checks
assert isinstance(eq, np.ndarray), "Output must be a numpy array"
assert eq.dtype == np.uint8, "Output dtype must be uint8"
assert eq.shape == im.shape, "Output must keep the same shape as input"

# Properties we expect from equalization on a bimodal image:
# - full or near-full contrast stretch (min 0, max 255)
# - monotonic mapping: darker inputs map to lower outputs than brighter inputs
assert eq.min() == 0, "Equalized image should start at 0"
assert eq.max() == 255, "Equalized image should reach 255"
assert len(np.unique(eq)) >= 2, "Equalized image should have at least two distinct levels"

# Monotonicity on representative pixels
low_out  = eq[0, 0]     # came from 10
high_out = eq[-1, -1]   # came from 200
assert low_out < high_out, "Equalization must preserve order (low should map below high)"

print("✅ histeq public tests passed!")

✅ histeq public tests passed!


Let's apply our function to the test image and compare:

In [224]:
# Apply our equalization function
img_dark_eq = histeq(img_dark)

# Calculate the histograms again to verify
hist_dark_eq = np.zeros(256)
for i in range(256):
    hist_dark_eq[i] = np.sum(img_dark_eq == i)

In [225]:
# Display the original and equalized images side by side
display_images(img_dark, img_dark_eq, "Original Image", "Equalized Image")

print("Original image range:", img_dark.min(), "-", img_dark.max())
print("Equalized image range:", img_dark_eq.min(), "-", img_dark_eq.max())

<IPython.core.display.Javascript object>

Original image range: 0 - 100
Equalized image range: 0 - 255


In [226]:
# Compare the histograms
plt.figure(figsize=(14,5))

plt.subplot(1, 2, 1)
plt.bar(range(256), hist_dark)
plt.title('Original Histogram')
plt.xlim([0, 255])

plt.subplot(1, 2, 2)
plt.bar(range(256), hist_dark_eq)
plt.title('Equalized Histogram')
plt.xlim([0, 255])

plt.tight_layout()

<IPython.core.display.Javascript object>

In [227]:
# Now let's try it with our real grayscale image
ph2_eq = histeq(ph2)

# Display the original and equalized grayscale images
display_images(ph2, ph2_eq, "Original Grayscale", "Equalized Grayscale")

<IPython.core.display.Javascript object>

Let's examine the histograms of the real image before and after equalization:

In [228]:
# Calculate histograms for the real image
hist_orig = np.zeros(256)
hist_eq = np.zeros(256)

for i in range(256):
    hist_orig[i] = np.sum(ph2 == i)
    hist_eq[i] = np.sum(ph2_eq == i)

# Plot histograms
plt.figure(figsize=(14,5))

plt.subplot(1, 2, 1)
plt.bar(range(256), hist_orig)
plt.title('Original Histogram')
plt.xlim([0, 255])

plt.subplot(1, 2, 2)
plt.bar(range(256), hist_eq)
plt.title('Equalized Histogram')
plt.xlim([0, 255])

plt.tight_layout()

<IPython.core.display.Javascript object>

In [229]:
# Let's see how the transformation function (CDF) looks
plt.figure()
plt.plot(np.arange(256), np.cumsum(hist_orig)/np.sum(hist_orig)*255)
plt.title('Transformation Function (CDF)')
plt.xlabel('Input pixel value')
plt.ylabel('Output pixel value')

<IPython.core.display.Javascript object>

Text(0, 0.5, 'Output pixel value')

Observations:
- The original image has pixels concentrated in the darker range
- The equalized image spreads the pixel values across the entire range
- The equalization process significantly improves contrast

In [230]:
# Compare the numerical statistics
print("Original image statistics:")
print(f"Min: {ph2.min()}, Max: {ph2.max()}, Mean: {ph2.mean():.2f}, Std: {ph2.std():.2f}")
print("\nEqualized image statistics:")
print(f"Min: {ph2_eq.min()}, Max: {ph2_eq.max()}, Mean: {ph2_eq.mean():.2f}, Std: {ph2_eq.std():.2f}")

Original image statistics:
Min: 0, Max: 255, Mean: 96.05, Std: 89.33

Equalized image statistics:
Min: 0, Max: 255, Mean: 130.08, Std: 75.01


In [231]:
# Let's also zoom in on a specific region to see the details better
y_start, y_end = 100, 300
x_start, x_end = 200, 500

region_orig = ph2[y_start:y_end, x_start:x_end]
region_eq = ph2_eq[y_start:y_end, x_start:x_end]

plt.figure(figsize=(14,6))

plt.subplot(1, 2, 1)
plt.imshow(region_orig, cmap='gray')
plt.title('Original Image (Zoomed)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(region_eq, cmap='gray')
plt.title('Equalized Image (Zoomed)')
plt.colorbar()

plt.tight_layout()

<IPython.core.display.Javascript object>

Key observations about histogram equalization:
- It increases global contrast, especially when the image has close contrast values
- Details in dark/bright regions become more visible
- The algorithm is non-linear and can sometimes produce unnatural looking images
- Works best on images with poor initial contrast

## Image filtering

Many image and video applications rely on two dimensional filters.

We will use an operation called convolution to apply the filters to our data: $h * x$ .

Besides image and video processing, convolution operation is utlized in
- robotics/computer vision,
- audio processing,
- AI: convolutional neural networks utilize the sibling operation: cross-correlation),
- statistics, etc.


The naive convolution implementation requires 4 nested `for` loops and is therefore very computationally intensive.

Convolution operation requires folding, shifting, summation and multiplication. The above formula includes all these operations.

For image processing, we have to fold twice: up-down and left-right.

See __[Wiki: Visual explanation of convolution](https://en.wikipedia.org/wiki/Convolution#Visual_explanation)__ for more info.

In terms of implementating convolution for image filtering, __[example Wiki algorithm](https://en.wikipedia.org/wiki/Kernel_%28image_processing%29)__ might be helpful:



"For example, if we have two three-by-three matrices, the first a kernel (our filter), and the second an image piece, convolution is the process of flipping both the rows and columns of the kernel and then multiplying locally similar entries and summing.

The element at coordinates [2, 2] (that is, the central element) of the resulting image would be a weighted combination of all the entries of the image matrix, with weights given by the kernel:
$$
\left(
\begin{bmatrix}
a & b & c \\
d & e & f \\
g & h & i
\end{bmatrix}
*
\begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9
\end{bmatrix} \right) [2,2]
=
(i \cdot 1)+(h \cdot 2)+(g \cdot 3)+(f \cdot 4)+(e \cdot 5)+(d \cdot 6)+(c \cdot 7)+(b \cdot 8)+(a \cdot 9).
$$
"

Again, notice that:
- we sum all of these multiplications to obtain a single value
- kernel matrix (our filter coefficients) on the left is being processed bottom to top and right to left
- see the pseudocode at __[wiki example algorithm](https://en.wikipedia.org/wiki/Kernel_%28image_processing%29)__ for more hints:

``` python
for each image row in input image:
   for each pixel in image row:
      set accumulator to zero
      for each kernel row in kernel:
         for each element in kernel row:

            if element position  corresponding* to pixel position :
               multiply element value  corresponding* to pixel value
               add result to accumulator
            endif

      set output image pixel to accumulator
```
*corresponding input image pixels are found relative to the kernel's origin.


List of kernels commonly used for image processing are defined below as h0-h7

In [232]:

#Delta filter
h0 = np.array([[0., 0., 0.],[0., 1., 0.],[0, 0, 0]])

#Lowpass filter
h1 = np.array([[1., 1., 1.],[1, 1, 1],[1, 1, 1]])*(1/9)

#Highpass filter
h2 = np.array([[-1., -4., -1.],[-4, 26, -4],[-1, -4, -1]])*(1/6)

#Sobel filter
h3 = np.array([[1., 0., -1.],[2, 0, -2],[1, 0, -1]])

#Prewitt filter
h4 = np.array([[1., 0., -1.],[1, 0, -1],[1, 0, -1]])

#Laplacian filter
h5 = np.array([[1., 4., 1.],[4, -20, 4],[1, 4, 1]])

#Embos filter
h6 = np.array([[-4., -4., 0.],[-4, 1, 4],[0, 4, 4]])

#Engrave filter
h7 = np.array([[-1., 0., 0.],[0, 2, 0],[0, 0, 0]])

print(h7)

[[-1.  0.  0.]
 [ 0.  2.  0.]
 [ 0.  0.  0.]]


In [233]:
#Scipy has a built-in convolve function that you can use to verify your code:
from scipy import ndimage

f0 = ndimage.convolve(ph2.squeeze(), h0, mode='constant', cval=0.0)
#let's keep the original image shape and zero-pad all the out-of-index areas.

In [234]:
#For instance: h0, Delta kernel, should return the original image:
np.max(ph2.squeeze() - f0)

np.uint8(0)

In [235]:
pic2=ph2.squeeze()
#try h0 to h7.
#What is the diff between h3 and h6?
hx = h3
z1 = ndimage.convolve(pic2, hx, mode='constant', cval=0.0)

plt.figure()
imgplot = plt.imshow(z1, cmap='gray')


<IPython.core.display.Javascript object>

## Implementing 2D Convolution

Implement the 2D convolution algorithm as defined via the math formula or the pseudocode above.

### Edge Case Handling
When filtering images with convolution, you need to handle pixels near the edges of the image. There are several approaches:
1. **Zero padding**: Assume pixels outside the image boundary are zero (we'll use this)
2. **Border replication**: Duplicate the edge pixels
3. **Mirror reflection**: Reflect the image at the borders

### Notes on Implementation
- Focus on correctness rather than efficiency for this assignment
- Be careful with indexing - the kernel needs to be flipped (reversed) for true convolution
- Expect your implementation to be slower than built-in functions like `ndimage.convolve`
- Pay attention to the dimensions of your input and output arrays

In [236]:
def conv(kernel, pic):
    ''' 2D Convolution Implementation

    Parameters:
    -----------
    kernel : 2D numpy array
        The convolution filter/kernel to apply
    pic : 2D numpy array
        Grayscale image to be filtered

    Returns:
    --------
    z : 2D numpy array
        Filtered image (same dimensions as input)
    '''
    # step 1, ensure input is a 2D array via squeeze
    pic = np.squeeze(pic)
    
    # step 2, get dimensions of image and kernel
    pic_h, pic_w = pic.shape
    kernel_h, kernel_w = kernel.shape
    
    # step 3, initialize output arr with 0s, utilizing floats for calculations.
    z = np.zeros_like(pic, dtype=np.float64)
    
    # step 4, calculate the center coords of kernel.
    center_h = kernel_h // 2
    center_w = kernel_w // 2
    
    # step 5, loop through each pixel in output image
    for i in range(pic_h):
        for j in range(pic_w):
            
            # step 5a, init accumulator
            accumulator = 0.0
            
            # step 5b, loop through each kernel element
            for m in range(kernel_h):
                for n in range(kernel_w):
                    
                    # step 5b.1 calculate corresponding position in image
                    row = i - (m - center_h)
                    col = j - (n - center_w)
                    
                    # step 5b.2 Cceck if the calculated position is within image bounds.
                    if 0 <= row < pic_h and 0 <= col < pic_w:
                        
                        # step 5b.iii multiply image pixel by kernel element, and add to sum
                        accumulator += pic[row, col] * kernel[m, n]
            
            
            # step 5c store final convoluted value in array
            z[i, j] = accumulator
            
    # step 6 return
    return z

In [237]:
pic = ph2[0:5,0:10]
pic = np.arange(1,10).reshape(3,3)
print(pic.shape, ph2.shape)
M,N = pic.shape
for m in pic:
    print(m)

(3, 3) (732, 1080)
[1 2 3]
[4 5 6]
[7 8 9]


In [238]:
#convolution test area. You can experiment/debug your code here.
pic = np.arange(1,10).reshape(3,3)
z = conv(h0, pic)
z1 = ndimage.convolve(pic, h0, mode='constant', cval=0.0)
print(z)

[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]


In [239]:
"""Verify that the convolution implementation is correct"""
#h0 (delta) is a special case filter
z = conv(h0, pic)
z1 = ndimage.convolve(pic, h0, mode='constant', cval=0.0)
print(z)
print(z1)

[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [240]:
"""If you need to debug the implementation, you can use more test cases"""
#h1 (lowpass) is another easy test case
z = conv(h1, pic)
z1 = ndimage.convolve(pic, h1, mode='constant', cval=0.0)
print("Custom implementation:", z)
print("Built-in function:", z1)

Custom implementation: [[1.33333333 2.33333333 1.77777778]
 [3.         5.         3.66666667]
 [2.66666667 4.33333333 3.11111111]]
Built-in function: [[1 2 1]
 [2 5 3]
 [2 4 3]]


In [241]:
"""Check that method returns the correct output for the photo"""
pic2=ph2
#try h0 to h7
hx = h0
z = conv(hx, pic2)
z1 = ndimage.convolve(pic2.squeeze(), hx, mode='constant', cval=0.0)

np.testing.assert_allclose(z, z1, rtol=1)


In [242]:
# === PUBLIC TESTS: conv ===
import numpy as np

# Helper to build an impulse image
def impulse(h, w, y, x):
    img = np.zeros((h, w), dtype=float)
    img[y, x] = 1.0
    return img

# --- Test 1: Identity kernel should return the same image ---
identity = np.array([[0, 0, 0],
                     [0, 1, 0],
                     [0, 0, 0]], dtype=float)

img = np.arange(25, dtype=float).reshape(5, 5)
out = conv(identity, img)

assert isinstance(out, np.ndarray), "Output must be a numpy array"
assert out.shape == img.shape, "Output must be the same shape as input"
assert np.allclose(out, img), "Convolution with identity kernel should return the original image"

# --- Test 2: Impulse image should return (flipped) kernel centered in output ---
# Use a symmetric kernel so flip == kernel (avoids sign/flip confusion in a public test)
kernel = np.ones((3, 3), dtype=float) / 9.0
img2   = impulse(7, 7, 3, 3)  # center impulse

out2 = conv(kernel, img2)

# Expected: kernel placed at the center, zeros elsewhere (with zero padding)
exp = np.zeros_like(out2)
exp[2:5, 2:5] = kernel  # centered block

assert np.allclose(out2, exp), "Impulse response should equal the kernel centered in the output"

print("✅ conv public tests passed!")


✅ conv public tests passed!


In [243]:
# Compare performance between your implementation and the built-in function

import time

# Prepare the image and kernel for a realistic test
test_image = ph2.squeeze()
test_kernel = h3  # Sobel filter - a common edge detection kernel

# Measure time for your implementation
start_time = time.time()
your_result = conv(test_kernel, test_image)
your_time = time.time() - start_time
print(f"Your implementation: {your_time:.4f} seconds")

# Measure time for built-in function
start_time = time.time()
builtin_result = ndimage.convolve(test_image, test_kernel, mode='constant', cval=0.0)
builtin_time = time.time() - start_time
print(f"Built-in function: {builtin_time:.4f} seconds")
print(f"Speed ratio: Your implementation is ~{your_time / builtin_time:.1f}x slower")

# Calculate the difference
diff = your_result - builtin_result
max_diff = np.max(np.abs(diff))
print(f"Maximum absolute difference between implementations: {max_diff:.6f}")

# Display your result
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(your_result, cmap='gray')
plt.title('Your Implementation')
plt.colorbar()

# Display built-in result
plt.subplot(1, 3, 2)
plt.imshow(builtin_result, cmap='gray')
plt.title('Built-in Function')
plt.colorbar()

# Display difference (scaled for visibility)
plt.subplot(1, 3, 3)
plt.imshow(np.abs(diff), cmap='hot')
plt.colorbar()
plt.title('Absolute Difference')

plt.tight_layout()

Your implementation: 2.1488 seconds
Built-in function: 0.0061 seconds
Speed ratio: Your implementation is ~350.4x slower
Maximum absolute difference between implementations: 1024.000000


<IPython.core.display.Javascript object>

Now let's try applying different kernels to see their effects:

In [244]:
"""Check that method returns the correct output for the photo"""
pic2=ph2
#try h0 to h7
hx = h0
z = conv(hx, pic2)
z1 = ndimage.convolve(pic2.squeeze(), hx, mode='constant', cval=0.0)

np.testing.assert_allclose(z, z1, rtol=1)

## Debugging hints

We can use Python's pdb debugger in Jupyter Notebooks.

This is an old school, command line debugger.

```python
from IPython.core.debugger import set_trace
```
Add *set_trace()* in your method to add a breakpoint

Useful pdb commands:

- pp <varname> : pretty print var. pp i
- c: continue until next breakpoint
- s: step in
- n: next line
- w: where, print stack trace
- q: quit debugger
    
Docs:
https://docs.python.org/3/library/pdb.html

## Grading Rubric (100 points total)

### White Balance Function (20 points)
- Correctly combines RGB arrays into 3D matrix (5 points)
- Proper normalization of values (5 points)
- Correct type conversion to uint8 (5 points)
- Results visually improve image white balance (5 points)

### Grayscale Conversion (15 points)
- Correctly implements conversion formula (5 points)
- Handles array dimensions properly (5 points)
- Code optimized without unnecessary loops (5 points)

### Histogram Equalization (25 points)
- Correctly calculates CDF (5 points)
- Proper implementation of normalization (5 points)
- Correct handling of edge cases (5 points)
- Results show appropriate contrast enhancement (10 points)

### Convolution Implementation (30 points)
- Correct implementation of convolution algorithm (10 points)
- Proper kernel flipping for convolution (5 points)
- Correct handling of image boundaries (5 points)
- Results match expected output from built-in functions (10 points)

### Code Quality (10 points)
- Code is well-commented and readable (3 points)
- Variable names are descriptive (2 points)
- Proper use of numpy functions and operations (3 points)
- No unnecessary code or debug statements (2 points)

**Note**: Partial credit may be given for incomplete implementations that demonstrate understanding of the concepts.

## Reflection Questions

After completing the assignment, please answer the following questions (2-3 sentences each):

1. How do the different convolution kernels (h0-h7) affect the image differently? Which one did you find most useful for enhancing details?

2. Compare the computational complexity of your convolution implementation with the built-in scipy function. Why is there a difference in performance?

3. How could the algorithms you implemented be applied to real-world problems beyond simple image enhancement?

4. How does this work relate to modern deep learning approaches in computer vision? Research and briefly explain the connection between convolution as implemented here and convolutional neural networks (CNNs).

1. Different kernels act like different types of filters, changing what the image look like. For example, the low pass filter, or h1, averages pixel values which blurs the image. Sobel (h3) and Prewitt (h4) kernels are edge detection kernels, which creates an outline around objects in the image. h5, or laplacian kernel, was also an edge detection kernel. Also of note, h6 affects the image by providing an embossed like effect. Next, h7, or the engrave kernel, acts like a sharpening kernel which makes the image look like it was engraved onto something. Finally, the kernel that was the best at enhancing details was the high pass filter, or h2, which amplifies the difference between 2 pixels, sharpening the image.
2. My manual implentation exhibits O(N^2*K^2), where the image is N and kernel is K/ This is due to the nested loops utilized in the solution. Compare this to scipy, where the fast fourier transform is used and has an O(n^2 log n) complexity, being much faster than mine. This difference in performance is caused by the fact that loops are an inherently slow process, whereas completing it in fourier transforms will massively accelerate the process.
3. These algorithms could be imposed upon the medical field, helping with improving contrast in x-rays, MRIs, and other types of scans. Also, edge detection algorithms can be used in the autonomous driving field, to help identify lane markings and potential hazards in the roadway.
4. This relates to deep learning approaches as convolution is the inherent backbone of convolutional neural networks. The primary difference between this implementation of convolution and a CNN's implementation is that this implementation utilizes manually specified kernel values for its fixed & singular task, whereas a CNN's implementation learns it's optimal kernel values over time, and utilizes several layers of convolutions to complete more complex tasks.
